# Convert Raw TEP Data

Read each raw `.RData` file and write its first object to `data/interim/TEP/` as Parquet.

In [1]:
from pathlib import Path
import sys


def _find_carl_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = _find_carl_project_root()
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
from pathlib import Path

import pyreadr


def find_project_root(start: Path = Path.cwd()) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "environment.yml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "TEP"
INTERIM_DATA_PATH = PROJECT_ROOT / "data" / "interim" / "TEP"
INTERIM_DATA_PATH.mkdir(parents=True, exist_ok=True)

if not RAW_DATA_PATH.is_dir():
    raise FileNotFoundError(f"Raw data directory does not exist: {RAW_DATA_PATH}")

print(f"Project root: {PROJECT_ROOT}")

Project root: <repo>


In [3]:
raw_data_files = sorted(RAW_DATA_PATH.glob("*.RData"))
if not raw_data_files:
    raise FileNotFoundError(f"No .RData files found in {RAW_DATA_PATH}")

for raw_data_file in raw_data_files:
    r_objects = pyreadr.read_r(str(raw_data_file))
    if not r_objects:
        raise ValueError(f"No R objects found in {raw_data_file.name}")

    dataframe = next(iter(r_objects.values()))
    output_path = INTERIM_DATA_PATH / raw_data_file.with_suffix(".parquet").name
    dataframe.to_parquet(output_path)
    print(f"Wrote {output_path.name}: {dataframe.shape}")

Wrote TEP_FaultFree_Testing.parquet: (480000, 55)
Wrote TEP_FaultFree_Training.parquet: (250000, 55)
Wrote TEP_Faulty_Testing.parquet: (9600000, 55)
Wrote TEP_Faulty_Training.parquet: (5000000, 55)
